In [36]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score
from src.preprocessing import train_valid_test_split
from src.evaluate import precision_k, score_precision_10
from src.feat_eng import add_log1p_feat
from sklearn.model_selection import GridSearchCV

In [47]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
train, valid, test = train_valid_test_split(df)
k = int(len(valid) * 0.1)

In [49]:
train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
train['pdays'] = train['pdays'].replace(-1, np.nan)

valid['pdays_prev_camp'] = valid['pdays'].ne(-1).astype('int8')
valid['pdays'] = valid['pdays'].replace(-1, np.nan)

# cols_to_log = ['campaign', 'previous', 'pdays']
#
# train = add_log1p_feat(train, cols_to_log)
# valid = add_log1p_feat(valid, cols_to_log)
#
# train['avg_balance'] = (
#     train.groupby('job')['balance'].transform('mean')
# )
# valid['avg_balance'] = (
#     valid.groupby('job')['balance'].transform('mean')
# )

train['job_freq'] = (
    train.groupby('job')['job'].transform('count') / train.job.count()
)
valid['job_freq'] = (
    valid.groupby('job')['job'].transform('count') / valid.job.count()
)


/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_19217/3033313558.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays_prev_camp'] = train['pdays'].ne(-1).astype('int8')
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_19217/3033313558.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['pdays'] = train['pdays'].replace(-1, np.nan)
/var/folders/d5/hmchxs4s0v7bc772rvxtlcdw0000gn/T/ipykernel_19217/3033313558.py:4: SettingWithCopyWarning: 
A value is trying to be set on a 

In [50]:
X_train = train.drop(columns=['y', 'duration'])
y_train = train["y"]
X_valid = valid.drop(columns=['y', 'duration'])
y_valid = valid['y']

In [51]:

cat_cols = X_train.select_dtypes(
    include=["object", "category", "string", "bool"]
)
cat_indices = [
      i for i, col in enumerate(X_train.columns)
      if X_train[col].dtype.name in ("object", "string", "category")
        ]

# eval_pool = Pool(X_valid, y_valid, cat_features=cat_indices)

In [52]:
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.01,
    loss_function="Logloss",
    depth=8,
    random_seed=0,
    l2_leaf_reg=3,
)
model.fit(X_train, y_train, cat_features=cat_indices, verbose=100)

0:	learn: 0.6808053	total: 36.2ms	remaining: 36.2s
100:	learn: 0.2521329	total: 2.46s	remaining: 21.9s
200:	learn: 0.2114865	total: 5.05s	remaining: 20.1s
300:	learn: 0.2020921	total: 8.16s	remaining: 18.9s
400:	learn: 0.1980142	total: 13.6s	remaining: 20.3s
500:	learn: 0.1956061	total: 18.1s	remaining: 18s
600:	learn: 0.1939998	total: 21.4s	remaining: 14.2s
700:	learn: 0.1927059	total: 24.6s	remaining: 10.5s
800:	learn: 0.1914029	total: 29.2s	remaining: 7.26s
900:	learn: 0.1899734	total: 41.8s	remaining: 4.59s
999:	learn: 0.1884723	total: 45.6s	remaining: 0us


CatBoostClassifier(depth=8, iterations=1000, l2_leaf_reg=3, learning_rate=0.01, loss_function='Logloss', random_seed=0)

In [42]:
# X_search = pd.concat([X_train, X_valid], ignore_index=True)
# y_search = pd.concat([y_train, y_valid], ignore_index=True)
#
# param_grid = {
#       "depth": [4, 6, 8],
#       "learning_rate": [0.01, 0.03, 0.1],
#       "l2_leaf_reg": [3, 10],
# }
#
# grid = GridSearchCV(estimator=model,
#     param_grid=param_grid,
#     cv=[(np.arange(len(X_train)), np.arange(len(X_train), len(X_search)))],
#     scoring=score_precision_10,
#     refit=False,
#     n_jobs=1,
#     error_score='raise',
# )
#
# grid.fit(X_search, y_search, cat_features=cat_indices, verbose=100)


In [53]:
y_pred = model.predict(X_valid)

In [54]:
yes_idx = list(model.classes_).index("yes")
y_score = model.predict_proba(X_valid)[:, yes_idx]
prec_k = precision_k(y_valid, y_score, k=k)
avg_precision = average_precision_score(y_valid, y_score, pos_label="yes")
# avg_precision = model.eval_metrics(data=eval_pool, metrics='Precision')

In [55]:
pd.Series({
    "precision_at_10pct": prec_k,
    "average_precision": avg_precision,
})

precision_at_10pct    0.275811
average_precision     0.234087
dtype: float64

#### exp 1:
- depth 6, iteration 500, random seed = 0, logloss, lr = 0.01
- metrics: prec_k = 0.2743, avg_prec = 0.2322

#### exp 2:
- same model pars
- metrics: prec_k = 0.2935, avg_prec = 0.2361

#### exp 3:
- {'depth': 8, 'l2_leaf_reg': 3, 'learning_rate': 0.01} - best params
- metrics: prec_k = 0.2979, avg_prec = 0.2388

#### exp 4:
- best params, logp1 of 'campaign', 'previous', 'pdays' features
- prec_k = 0.2669, avg_prec = 0.2213
- revert to exp 3

#### exp 5:
- best params, add feature 'avg_balance' by job
- prec_k = 0.2581, avg_prec = 0.2271
- revert to exp 3

#### exp 6:
- best params, add feature 'job_freq'
- precision_at_10pct = 0.2758, average_precision = 0.2340
- revert to exp 3
